# Covariate-shift conformal prediction

Use pooled detector features to fit a source-versus-target classifier, then weight the internal true-class scores. External labels are not passed to calibration.

## Load the data

Place the downloaded files in `data/`, or set `GFLD_DATA_DIR` to their location. Dataset details are provided in the paper.

In [ ]:
%matplotlib inline
from pathlib import Path
from dataclasses import asdict
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from gfld import *

# Works when Jupyter starts in the repository root or notebooks directory.
ROOT = Path.cwd() if (Path.cwd() / "pyproject.toml").is_file() else Path.cwd().parent
DATA_DIR = example_directory(ROOT)  # GFLD_DATA_DIR or repository data/
data = load_example(DATA_DIR)
print({k: v["images"] for k, v in data["metadata"]["splits"].items()})

In [ ]:
# Split calibration and evaluation by image.
_, calibration_idx, evaluation_idx = next(calibration_splits(
    len(data["internal"]), folds=5, permutations=1, seed=42))
calibration = [data["internal"][i] for i in calibration_idx]
evaluation = [data["internal"][i] for i in evaluation_idx]
annotations = data["internal_annotations"]
assert set(p.image_id for p in calibration).isdisjoint(p.image_id for p in evaluation)
print(f"{len(calibration)} calibration images; {len(evaluation)} held-out images")

In [ ]:
cp = calibrate_cp(calibration, annotations, alpha=0.01)
pd.DataFrame([asdict(cp)])

## Estimate weights and recalibrate

Fit a source-versus-target logistic classifier using image features. Apply the resulting clipped weights to the class-score quantile.

In [ ]:
weighted_cp, cp_weights, domain_model = calibrate_shift_cp(
    calibration, annotations, data["internal_features"], data["external_features"],
    alpha=0.01, weight_cap=10.0)
pd.DataFrame([asdict(cp), asdict(weighted_cp)], index=["ordinary", "weighted"])

In [ ]:
plt.hist(list(cp_weights.values()), bins=30)
plt.xlabel("Clipped image weight"); plt.ylabel("Matched source images")
plt.show()
print(pd.Series(cp_weights).describe())

## Evaluate on external images

Compare ordinary and weighted label sets using the external annotations.

In [ ]:
external = data["external"]
external_annotations = data["external_annotations"]
pd.DataFrame([evaluate_cp(external, external_annotations, cp),
              evaluate_cp(external, external_annotations, weighted_cp)],
             index=["ordinary CP", "weighted CP"])

## Compare label-set sizes

Summarize the number of classes assigned to each external detection.

In [ ]:
sizes = [np.concatenate([model.predict(p.class_scores).sum(axis=1) for p in external])
         for model in (cp, weighted_cp)]
plt.hist(sizes, bins=np.arange(-0.5, 4.5), label=["ordinary", "weighted"])
plt.xticks(range(4)); plt.xlabel("Classes per label set"); plt.ylabel("External detections")
plt.legend(); plt.show()